# Invoice Field Extractor

**NLP Information Extraction — Parse invoice number, vendor, totals, dates from invoice text**

## Project Overview

Extract structured fields from raw invoice text using **regex** and **spaCy NER**.

## Learning Objectives

1. Combine regex and NER for structured extraction.
2. Handle semi-structured document formats.
3. Validate extraction with field-level accuracy.

## Problem Statement

Extract invoice number, date, vendor, total from raw invoice text.

## Why This Project Matters

- AP teams process thousands of invoices manually.
- Extraction errors cause payment delays.
- Automating extraction saves labor and reduces errors.

## Dataset Overview

100 synthetic invoices with known ground-truth fields.

## Dataset Source and License Notes

Synthetic data.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
VENDORS = ["Acme Corp","GlobalTech Inc","Smith & Co","Delta Services",
           "Pacific Supply Ltd","Northwind Traders","Contoso Ltd",
           "Fabrikam Inc","Alpine Solutions","Meridian Group"]
rng = np.random.default_rng(SEED)
print(f"Vendors: {len(VENDORS)}")


## Dataset Download or Loading — Generate Synthetic Invoices

In [ ]:
invoices = []
for i in range(100):
    inv_num = f"INV-{rng.integers(10000,99999)}"
    vendor = rng.choice(VENDORS)
    date = f"{rng.integers(2024,2026)}-{rng.integers(1,13):02d}-{rng.integers(1,29):02d}"
    subtotal = round(rng.uniform(100, 50000), 2)
    tax = round(subtotal * 0.08, 2)
    total = round(subtotal + tax, 2)
    text = f"Invoice Number: {inv_num}\nDate: {date}\nFrom: {vendor}\nSubtotal: ${subtotal:,.2f}\nTax: ${tax:,.2f}\nTotal Due: ${total:,.2f}\nPayment Terms: Net 30"
    invoices.append({"text": text, "true_inv_num": inv_num, "true_vendor": vendor, "true_date": date, "true_total": total})
df = pd.DataFrame(invoices)
print(f"Generated {len(df)} invoices")
print(df["text"].iloc[0])


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
assert df["true_inv_num"].str.startswith("INV-").all()
print(f"Validated: {len(df)} invoices")


## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df["true_total"],bins=30,color="steelblue",edgecolor="white")
axes[0].set_title("Invoice Total Distribution")
df["true_vendor"].value_counts().plot.bar(ax=axes[1],color="coral")
axes[1].set_title("Invoices per Vendor")
plt.tight_layout(); plt.show()


## Target Analysis

4 fields: invoice number, date, vendor, total.

## Train/Validation/Test Split Strategy

Not applicable — rule-based extraction.

## Preprocessing Strategy

Regex patterns for each field.

## Feature Engineering — Extraction Functions

In [ ]:
def extract_invoice_number(text):
    m = re.search(r"Invoice\s*(?:Number|No\.?|#)?\s*:?\s*(INV-?\d+)", text, re.IGNORECASE)
    return m.group(1) if m else None
def extract_date(text):
    m = re.search(r"Date:\s*(\d{4}-\d{2}-\d{2})", text)
    return m.group(1) if m else None
def extract_total(text):
    m = re.search(r"Total\s*Due:\s*\$([\d,]+\.\d{2})", text)
    return float(m.group(1).replace(",","")) if m else None
def extract_vendor(text):
    m = re.search(r"From:\s*(.+)", text)
    return m.group(1).strip() if m else None
sample = df["text"].iloc[0]
print(f"Invoice #: {extract_invoice_number(sample)}")
print(f"Total:     {extract_total(sample)}")


## Baseline Model — Run Extraction

In [ ]:
df["ext_inv_num"] = df["text"].apply(extract_invoice_number)
df["ext_date"] = df["text"].apply(extract_date)
df["ext_vendor"] = df["text"].apply(extract_vendor)
df["ext_total"] = df["text"].apply(extract_total)
print("Field-level accuracy:")
for field in ["inv_num","date","vendor","total"]:
    acc = (df[f"true_{field}"] == df[f"ext_{field}"]).mean()
    print(f"  {field:10s}: {acc:.1%}")


## LazyPredict Benchmark

> **Not applicable.** This is an NLP IE task. LazyPredict is not used.

## FLAML AutoML Run

> **Not applicable.** FLAML is not used for NLP IE tasks.

## Additional Best-Library Workflow — spaCy NER for Vendor

In [ ]:
def extract_vendor_spacy(text):
    doc = nlp(text)
    orgs = [e.text for e in doc.ents if e.label_ == "ORG"]
    return orgs[0] if orgs else None
df["ext_vendor_spacy"] = df["text"].apply(extract_vendor_spacy)
spacy_acc = sum(df["true_vendor"] == df["ext_vendor_spacy"]) / len(df)
regex_acc = (df["true_vendor"] == df["ext_vendor"]).mean()
print(f"Vendor — Regex: {regex_acc:.1%}, spaCy: {spacy_acc:.1%}")


## Top 3 Model Selection

Single pipeline. Regex vs spaCy NER for vendor field.

In [ ]:
fields = ["inv_num","date","vendor","total"]
accs = [(df[f"true_{f}"] == df[f"ext_{f}"]).mean() for f in fields]
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(fields, accs, color="steelblue")
ax.set_title("Field Extraction Accuracy"); ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()


## Final Training and Evaluation of Top 3

The regex pipeline achieves near-perfect accuracy on consistent formats.

## Error Analysis

In [ ]:
for field in ["inv_num","date","vendor","total"]:
    failures = df[df[f"true_{field}"] != df[f"ext_{field}"]]
    print(f"{field}: {len(failures)} failures")


## Interpretation and Business Insight

1. **Regex is highly effective** for consistent formats.
2. **spaCy NER supplements** regex for organization names.
3. **Real invoices vary much more** — need layout-aware models.
4. **Even simple automation** saves thousands of hours.

## Limitations

- Regex patterns are format-specific.
- No scanned/image invoice support.
- No line-item extraction.
- Single currency format only.

## How to Improve This Project

1. Add **OCR** (PaddleOCR) for scanned invoices.
2. Use **LayoutLM** for layout-aware extraction.
3. Extract **line items**.
4. Handle **multiple date formats**.

## Production Considerations

- Integrate with ERP/accounting systems.
- Add human-in-the-loop review.
- Handle multiple invoice templates.
- Log extraction results for audit.

## Common Mistakes

1. **Overfitting regex** to one format.
2. **Not handling currency** separators.
3. **Not validating** totals against line items.

## Mini Challenge / Exercises

1. Add line-item extraction.
2. Handle multiple date formats.
3. Add confidence scores.
4. Build a JSON output schema.

## Final Summary / Key Takeaways

1. **Regex** works well for consistent formats.
2. **spaCy NER** supplements regex.
3. **For production, consider LayoutLM** or PaddleOCR.
4. **Document IE** is a high-value enterprise application.